# Load Imports

In [167]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
import plotly.express as px
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score


# Binary classification

In this Notebook, we will complete the following tasks:
- Examine the dataset containing measurements.
- Create a binary classifier to sort drop outs.
- Evaluate the performance of the model.

## Objectives

By completing this notebook, you'll have:
- Plot feature on graphs
- Generate a csv (after_cleaning.csv) of cleaned dataset
- Train a binary classifier.
- Calculate metrics for a binary classifier at different thresholds.
- Compare AUC and ROC of two different models.

## Dataset

This Notebook uses the student droput prediction set dataset on kaggle.

Provided with a CC0 license (see [Kaggle](https://www.kaggle.com/datasets/meharshanali/student-dropout-prediction-dataset) for more documentation.)

In [168]:
df = pd.read_csv("student_dropout_dataset_v3.csv", index_col=0)
df.describe()


,Age,Family_Income,Study_Hours_per_Day,Attendance_Rate,Assignment_Delay_Days,Travel_Time_Minutes,Stress_Index,GPA,Semester_GPA,CGPA,Dropout
count,10000.00000,9500.000000,9500.000000,10000.00000,10000.000000,10000.00000,9500.000000,10000.000000,10000.000000,10000.000000,10000.00000
mean,21.02606,38377.247474,4.014592,81.73683,1.799700,30.17926,5.507147,2.308440,2.300057,2.298761,0.23540
std,2.13981,20496.232179,1.295450,8.22093,1.344307,11.91887,1.765951,1.061717,1.074407,1.072555,0.42427
min,17.00000,25000.000000,0.500000,38.20000,0.000000,5.00000,1.000000,0.000000,0.000000,0.000000,0.00000
25%,19.50000,25000.000000,3.160000,76.40000,1.000000,21.90000,4.300000,1.550000,1.520000,1.520000,0.00000
50%,21.00000,29740.500000,4.000000,81.80000,2.000000,30.20000,5.500000,2.350000,2.350000,2.350000,0.00000
75%,22.50000,44520.000000,4.870000,87.30000,3.000000,38.40000,6.700000,3.120000,3.150000,3.150000,0.00000
max,29.60000,316601.000000,8.980000,100.00000,8.000000,74.90000,10.000000,4.000000,4.000000,4.000000,1.00000


# Explore the dataset

Plot some of the features against each other, including in 3D.


In [169]:
y = df['Dropout']
X_full = df.drop(['Dropout'], axis=1)

# Color-code by a drop out
color_col = 'Dropout'  # alternatives: 'Gender', 'Semester', 'Parental_Education'

for x_axis_data, y_axis_data in [
    ('Study_Hours_per_Day', 'GPA'),
    ('Attendance_Rate', 'CGPA'),
    ('Stress_Index', 'GPA'),
    ('Assignment_Delay_Days', 'Semester_GPA'),
    ('Travel_Time_Minutes', 'Attendance_Rate'),
]:
    px.scatter(
        df,
        x=x_axis_data,
        y=y_axis_data,
        color=color_col
    ).show()

# Clean Dataset

## Duplicates
df.duplicated().sum()

## Normalize data

When creating a model with multiple features, the values of each feature should span roughly the same range.  If one feature's values range from 500 to 100,000 and another feature's values range from 2 to 12, the model will need to have weights of extremely low or extremely high values to be able to combine these features effectively. This could result in a low quality model.  To avoid this, we [normalize](https://developers.google.com/machine-learning/glossary/#normalization) features

This can be done by converting each raw value to its Z-score. The **Z-score** for a given value is how many standard deviations away from the mean the value is.

## Preprocessing pipeline
Here we bundle preprocessing and steps so you can use the whole bundle as if it were a single step

In [170]:
categorical_cols = [cname for cname in X_full.columns if X_full[cname].nunique() < 10 and X_full[cname].dtype == "string"]
numerical_cols = [cname for cname in X_norm.columns if X_norm[cname].dtype in ['float64']]
total_cols = categorical_cols + numerical_cols

# keep selected columns
X = X_full[total_cols].copy()

# Preprocessing for numerical data
numerical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
])
# Preprocessing for categorical data
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Bundle preprocessing for numerical and categorical data
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', categorical_transformer, categorical_cols)
])

X_after = preprocessor.fit_transform(X_train)

# Generate csv after cleaning
feature_names = preprocessor.get_feature_names_out()
X_after_df = pd.DataFrame(X_after, columns=feature_names)

X_after_df.to_csv('after_cleaning.csv', index=False)


# Set the random seeds

To make experiments reproducible, we set the seed of the random number generators. This means that the order in which the data is shuffled, the values of the random weight initializations, etc, will all be the same each time the notebook is run.

In [171]:
random_seed = 42

# Label and split data

To train the model, we'll arbritrarily assign the feature to X_full variable and target to y variable

In [172]:
X_train, X_valid, y_train, y_valid = train_test_split(X_full, y, test_size=0.2, random_state=random_seed)
X_valid, X_test, y_valid, y_test  = train_test_split(X_valid, y_valid, test_size=0.10, random_state=random_seed)

print(f"train: {X_train.shape}", f"valid: {X_valid.shape}", f"test: {X_test.shape}")

train: (8000, 17) valid: (1800, 17) test: (200, 17)


We can then randomize and partition the dataset into train, validation, and test splits, consisting of 80%, 10%, and 10% of the dataset respectively.

- Training data will be used to learn the model parameters
- Validation data is to evaluate different models
- Finally, test data to compute the final metrics for the model that performed best on the validation data.

# Model training


## Pipeline
Bundle preprocessing and modeling steps to use the whole bundle as if it were a single step

In [173]:
model_1 = LogisticRegression(random_state=random_seed)
model_2 = RandomForestRegressor(n_estimators=100, random_state=0)

# Bundle preprocessing and modeling code in a pipeline
my_pipeline = Pipeline(steps=[('preprocessor', preprocessor), ('model', model_1)])

# Preprocessing of training data, fit model 
my_pipeline.fit(X_train, y_train)
# preds = my_pipeline.predict(X_valid)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](17,)","['Age','Gender','Family_Income',...,'Semester','Department', 'Parental_Education']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,17
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrou

## Cross validation

In [175]:
scores = -1 * cross_val_score(my_pipeline, X, y,cv=5,scoring='neg_mean_absolute_error')
print(scores)

[0.183  0.1965 0.206  0.1775 0.18  ]
